# Persistent Storage & I/O for AI/Scientific Workloads

**Afternoon session · 12:15 – 12:45 PM** · [website version](https://training.nrp-nautilus.io/pearc26/4_storage.html) — run cells with **Shift+Enter**.


## ⚙️ Setup — run this first

Set your short username once; every command below uses `$NRP_USER`. The cell
also renders every manifest into **`my-yamls/`** with `<username>` already
filled in — wherever the website says *"replace `<username>`"*, it's already
done for you here.

> Terminal steps below don't share this variable — run the same
> `export NRP_USER=...` line in any terminal you open.
> Re-running this cell re-renders `my-yamls/` (overwriting any edits you made there).

**First time in one of these notebooks?** Click the **📌 pin icon** in the toolbar
above for a 30-second guided tour of how this notebook works.


In [ ]:
export NRP_USER=changeme   # ✏️ EDIT to your short name, then Shift+Enter
cd ~/pearc26/workspace
if [ "$NRP_USER" = changeme ]; then echo "⚠️  Edit NRP_USER above first, then re-run"; else
  mkdir -p my-yamls
  for f in yamls/*; do sed "s/<username>/$NRP_USER/g" "$f" > "my-yamls/$(basename "$f")"; done
  echo "✅ my-yamls/ rendered for $NRP_USER"
fi


**Afternoon session · 12:15 – 12:45 PM**

AI and scientific workloads live or die on I/O: where the dataset sits, how fast checkpoints write, and whether ten students can read the same files at once. This episode maps NRP's storage options to those needs and gets hands-on with each from a notebook terminal.

> 📘 **Docs:** [Storage intro](https://nrp.ai/documentation/userdocs/storage/intro/) · [Ceph](https://nrp.ai/documentation/userdocs/storage/ceph/) · [S3](https://nrp.ai/documentation/userdocs/storage/ceph-s3/) · [Policies](https://nrp.ai/documentation/userdocs/start/policies/)


## Storage options on NRP

| Option | Access mode | Best for | Caveats |
|---|---|---|---|
| **Pod filesystem** | per-container | scratch during a single run | gone when the pod dies |
| **`emptyDir`** | per-pod, shared by its containers | fast scratch, staging downloads | gone when the pod dies; counts against ephemeral-storage |
| **RBD block (`rook-ceph-block-*`)** | `ReadWriteOnce` | home dirs, checkpoints, databases | one pod at a time |
| **CephFS (`rook-cephfs-*`)** | `ReadWriteMany` | shared datasets, course materials, multi-pod pipelines | slightly slower metadata than block |
| **S3 (Ceph RGW)** | HTTP, from anywhere | dataset distribution, results publishing, cross-site access | object semantics, not POSIX |

Your JupyterHub home directory (`/home/jovyan`) is itself an RBD PVC — everything you save in the notebook survives server restarts, but it is sized in gigabytes; keep bulk data on CephFS or S3.


### Choosing an access mode

- **`ReadWriteOnce` (RWO)** — one node mounts read-write. Block storage. You saw the consequence in Episode 2: the sidecar exercise had to delete the first pod before the second could mount.
- **`ReadWriteMany` (RWX)** — many pods on many nodes mount simultaneously. CephFS. This is what a classroom shared folder or a multi-worker training job wants.


## Hands-on: an RWX CephFS volume shared by many pods

`yamls/shared-pvc.yaml` creates a CephFS-backed PVC. Apply it and note the `RWX` access mode:


In [ ]:
kubectl apply -n nrp-training-k8s -f my-yamls/shared-pvc.yaml
kubectl get pvc -n nrp-training-k8s | grep shared


<details>
<summary>Expected output</summary>

<pre style="line-height:1.45">
jupyterhub-shared-volume   Bound    pvc-…   5Gi   RWX   rook-cephfs   30s
</pre>
</details>


Unlike the Episode 2 PVC, **many pods can mount this claim at the same time** — in the final episode this exact volume becomes the `/home/shared` folder every student sees in a course JupyterHub.


## Hands-on: S3 object storage

NRP runs S3-compatible object storage on Ceph. It's the right tool when data must be reachable from outside the cluster, shared across sites, or published alongside a paper. Any S3 client works — `aws` CLI, `boto3`, `s3fs`, rclone.

`yamls/pod-awscli.yaml` starts a pod with the AWS CLI image, a 100 Gi `emptyDir` scratch volume at `/scratch`, and installs `boto3` + `torch` on boot. It also pulls the shared tutorial S3 credentials in from a Secret (`nrp-tutorial-s3`) as environment variables — so inside the pod both `aws` and `boto3` authenticate automatically, no `aws configure` step. Replace `<username>` and apply:


In [ ]:
kubectl apply -n nrp-training-k8s -f my-yamls/pod-awscli.yaml


**🖥️ Terminal step** — interactive or long-running: use a JupyterLab terminal (**File → New → Terminal**), not this notebook. Ctrl-C / Ctrl-D to exit.

```bash
kubectl get pod tutorial-$NRP_USER-pod -n nrp-training-k8s -w
```


Wait for the install loop to log `Done with installs`, then exec in and talk to S3:

**🖥️ Terminal step** — interactive or long-running: use a JupyterLab terminal (**File → New → Terminal**), not this notebook. Ctrl-C / Ctrl-D to exit.

```bash
kubectl exec -it tutorial-$NRP_USER-pod -n nrp-training-k8s -- bash
```


In [ ]:
# inside the pod — the tutorial key is already in the environment
# ($AWS_ACCESS_KEY_ID / $AWS_SECRET_ACCESS_KEY / $AWS_ENDPOINT_URL / $S3_BUCKET):
aws --endpoint $AWS_ENDPOINT_URL s3 ls
aws --endpoint $AWS_ENDPOINT_URL s3 ls s3://$S3_BUCKET/

# stage the shared dataset onto the fast local scratch:
aws --endpoint $AWS_ENDPOINT_URL s3 cp s3://$S3_BUCKET/dataset.tar.gz /scratch/
tar xzf /scratch/dataset.tar.gz -C /scratch     # 30 sample images + labels.csv + shards

# publish your own results back — write under your username so you don't
# clobber the shared dataset (everyone shares one bucket in this tutorial):
echo "hello from $NRP_USER" > /scratch/result.txt
aws --endpoint $AWS_ENDPOINT_URL s3 cp /scratch/result.txt s3://$S3_BUCKET/$NRP_USER/result.txt


> On your own laptop (outside this pod) you'd first run `aws configure` and paste
> an access key / secret from [nrp.ai/s3token](https://nrp.ai/s3token), then use the
> same `--endpoint` commands. Request your own S3 credentials via the User Portal.

The same works from Python with `boto3` — it reads the same credentials from the environment:

*(Python equivalent — run wherever your S3 credentials are configured:)*

```python
import boto3, os
s3 = boto3.client("s3", endpoint_url=os.environ["AWS_ENDPOINT_URL"])
bucket = os.environ["S3_BUCKET"]
for obj in s3.list_objects_v2(Bucket=bucket).get("Contents", []):
    print(obj["Key"], obj["Size"])
```


## I/O patterns for AI workloads

A pattern that serves nearly every training job on NRP:

1. **Stage in** — copy the dataset from S3 (or CephFS) to node-local scratch (`emptyDir`) at job start. Local NVMe is far faster for the random reads of a dataloader.
2. **Checkpoint out** — write checkpoints to an RWO block PVC (or push to S3) at epoch boundaries, not every step.
3. **Publish** — copy final artifacts to S3 where collaborators (or your future self) can fetch them without cluster access.

For classrooms, the equivalent pattern is: course materials on an **RWX CephFS volume** mounted read-only into every student server, student work on **per-user RWO home volumes** — exactly what we'll configure in the final episode.


## Cleanup


In [ ]:
kubectl delete pod tutorial-$NRP_USER-pod -n nrp-training-k8s --ignore-not-found


Keep `jupyterhub-shared-volume` — the custom JupyterHub episode mounts it. Verify with `bash check.sh 4`.


### 🧠 Quick check

**Ten student pods need to read the same course dataset at the same time. Which storage fits?**

- An RBD block PVC (`rook-ceph-block-east`)
- A CephFS PVC with ReadWriteMany
- An emptyDir volume

<details><summary><b>Reveal answer</b></summary>

**✔ A CephFS PVC with ReadWriteMany**

RWO block storage mounts on one node at a time (you hit that limit in the sidecar exercise). CephFS RWX mounts into many pods on many nodes — the classroom shared-folder pattern.

</details>

**Where should a training job stage its dataset for the fastest dataloader reads?**

- Read it straight from S3 every epoch
- Copy it once to node-local scratch (`emptyDir`) at job start
- Keep it in the home PVC

<details><summary><b>Reveal answer</b></summary>

**✔ Copy it once to node-local scratch (`emptyDir`) at job start**

Stage in → local NVMe scratch, checkpoint out → PVC or S3 at epoch boundaries, publish → S3. Local scratch is far faster for random reads than any network storage.

</details>

**What's true of NRP's S3 storage?**

- Reachable over HTTP from anywhere — object semantics, not a POSIX filesystem
- It mounts into pods like a normal directory
- Its contents disappear when your pod terminates

<details><summary><b>Reveal answer</b></summary>

**✔ Reachable over HTTP from anywhere — object semantics, not a POSIX filesystem**

S3 is the right tool for distribution and publishing: any S3 client works from inside or outside the cluster, but you `get`/`put` objects instead of doing POSIX file I/O.

</details>

**Your JupyterHub home directory (`/home/jovyan`) survives server restarts. What is it, really?**

- A per-user RWO block PVC — persistent, but sized in gigabytes
- Node-local disk on whichever node you spawned
- A CephFS share common to all users

<details><summary><b>Reveal answer</b></summary>

**✔ A per-user RWO block PVC — persistent, but sized in gigabytes**

Every hub user gets a `claim-<username>` RBD PVC as their home. It persists across sessions but it's small — keep bulk datasets on CephFS RWX volumes or S3, not in your home.

</details>

**During training, when and where should checkpoints be written?**

- To an RWO PVC or S3 at epoch boundaries — not every step
- To emptyDir scratch, for speed
- To stdout, so they end up in `kubectl logs`

<details><summary><b>Reveal answer</b></summary>

**✔ To an RWO PVC or S3 at epoch boundaries — not every step**

emptyDir dies with the pod — the one place a checkpoint must not live. Durable storage at epoch boundaries balances safety against I/O overhead: stage in → scratch, checkpoint out → PVC/S3, publish → S3.

</details>


---

## ✅ Check your work

Verifies the state of your resources on the cluster — rerun any time.


In [ ]:
bash check.sh 4
